# 🎬 CineIQ — Content-Based Filtering Module
## TMDB Movie Recommendation Engine using TF-IDF + Cosine Similarity

**Pipeline**: Hybrid Recommendation Engine → Content Filtering Block  
**Dataset**: Kaggle TMDB Metadata CSV Files  
**Technique**: TF-IDF Vectorization + Cosine Similarity  
**Goal**: Recommend similar movies based on metadata features

---

## 1. Environment Setup & Imports

In [ ]:
# ── Standard Library ──
import ast
import os
import re
import logging
import warnings
from typing import List, Optional, Dict, Any, Tuple
from difflib import get_close_matches

# ── Data & ML ──
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
import joblib

# ── Configuration ──
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 120)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('CineIQ_ContentFilter')
logger.info('All libraries imported successfully.')

## 2. Configuration & Constants

In [ ]:
# ── Paths ──
DATA_DIR = '.'  # Root folder containing CSVs (adjust if needed)
ARTIFACT_DIR = os.path.join(DATA_DIR, 'artifacts', 'content_filtering')

# ── TF-IDF Hyperparameters ──
TFIDF_MAX_FEATURES = 20_000
TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MIN_DF = 2
TFIDF_STOP_WORDS = 'english'

# ── Recommendation Defaults ──
DEFAULT_TOP_N = 10
TOP_CAST_COUNT = 3

# ── Create artifact directory ──
os.makedirs(ARTIFACT_DIR, exist_ok=True)
logger.info(f'Artifact directory ready: {ARTIFACT_DIR}')

## 3. Data Loading
Load the TMDB metadata CSV files. Supports both `.csv` and `.csv.zip` formats.

In [ ]:
def load_csv(filename: str, low_memory: bool = False) -> pd.DataFrame:
    """Load a CSV file, trying .csv.zip then .csv."""
    zip_path = os.path.join(DATA_DIR, f'{filename}.zip')
    csv_path = os.path.join(DATA_DIR, filename)
    if os.path.exists(zip_path):
        logger.info(f'Loading {zip_path}')
        return pd.read_csv(zip_path, low_memory=low_memory)
    elif os.path.exists(csv_path):
        logger.info(f'Loading {csv_path}')
        return pd.read_csv(csv_path, low_memory=low_memory)
    else:
        raise FileNotFoundError(f'Neither {zip_path} nor {csv_path} found.')

# Load datasets
df_meta = load_csv('movies_metadata.csv', low_memory=True)
df_credits = load_csv('credits.csv')
df_keywords = load_csv('keywords.csv')

print(f'movies_metadata : {df_meta.shape}')
print(f'credits         : {df_credits.shape}')
print(f'keywords        : {df_keywords.shape}')

## 4. Initial Data Inspection

In [ ]:
print('=== movies_metadata — first 3 rows ===')
display(df_meta.head(3))
print(f'\nNull counts:\n{df_meta.isnull().sum()[df_meta.isnull().sum() > 0]}')

In [ ]:
print('=== credits — first 2 rows ===')
display(df_credits.head(2))
print(f'\n=== keywords — first 2 rows ===')
display(df_keywords.head(2))

## 5. Data Preprocessing & Cleaning

### 5.1 Clean movies_metadata

In [ ]:
# Keep only relevant columns
META_COLS = ['id', 'title', 'overview', 'genres', 'vote_average',
             'vote_count', 'popularity', 'release_date']
df_meta = df_meta[META_COLS].copy()

# Remove rows where id is not numeric (corrupted rows)
df_meta['id'] = pd.to_numeric(df_meta['id'], errors='coerce')
rows_before = len(df_meta)
df_meta.dropna(subset=['id'], inplace=True)
df_meta['id'] = df_meta['id'].astype(int)
logger.info(f'Dropped {rows_before - len(df_meta)} rows with non-numeric id.')

# Drop rows missing title
df_meta.dropna(subset=['title'], inplace=True)

# Drop duplicates
df_meta.drop_duplicates(subset='id', keep='first', inplace=True)

# Fill missing overview
df_meta['overview'] = df_meta['overview'].fillna('')

# Parse release_date → year
df_meta['year'] = pd.to_datetime(df_meta['release_date'], errors='coerce').dt.year
df_meta['year'] = df_meta['year'].fillna(0).astype(int)

# Convert popularity and votes
df_meta['popularity'] = pd.to_numeric(df_meta['popularity'], errors='coerce').fillna(0.0)
df_meta['vote_average'] = pd.to_numeric(df_meta['vote_average'], errors='coerce').fillna(0.0)
df_meta['vote_count'] = pd.to_numeric(df_meta['vote_count'], errors='coerce').fillna(0).astype(int)

print(f'Cleaned movies_metadata shape: {df_meta.shape}')
display(df_meta.head(3))

### 5.2 Clean credits & keywords

In [ ]:
# Ensure id columns are int
df_credits['id'] = pd.to_numeric(df_credits['id'], errors='coerce')
df_credits.dropna(subset=['id'], inplace=True)
df_credits['id'] = df_credits['id'].astype(int)

df_keywords['id'] = pd.to_numeric(df_keywords['id'], errors='coerce')
df_keywords.dropna(subset=['id'], inplace=True)
df_keywords['id'] = df_keywords['id'].astype(int)

logger.info(f'credits shape: {df_credits.shape}, keywords shape: {df_keywords.shape}')

### 5.3 Merge Datasets

In [ ]:
# Merge on movie id
df = df_meta.merge(df_credits, on='id', how='left')
df = df.merge(df_keywords, on='id', how='left')

logger.info(f'Merged dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
display(df.head(3))

## 6. Feature Engineering
Build a combined metadata text field for TF-IDF vectorization.

### 6.1 JSON-like Column Parsers

In [ ]:
def safe_literal_eval(val: Any) -> Any:
    """Safely parse a JSON-like string using ast.literal_eval."""
    if isinstance(val, list):
        return val
    if pd.isna(val) or val == '':
        return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []


def extract_names(obj_list: Any, key: str = 'name', max_items: Optional[int] = None) -> List[str]:
    """Extract 'name' values from a list of dicts."""
    parsed = safe_literal_eval(obj_list) if not isinstance(obj_list, list) else obj_list
    names = [item[key] for item in parsed if isinstance(item, dict) and key in item]
    if max_items:
        names = names[:max_items]
    return names


def get_director(crew_data: Any) -> str:
    """Extract director name from crew list."""
    parsed = safe_literal_eval(crew_data) if not isinstance(crew_data, list) else crew_data
    for member in parsed:
        if isinstance(member, dict) and member.get('job') == 'Director':
            return member.get('name', '')
    return ''


def normalize_name(name: str) -> str:
    """Lowercase and remove spaces: 'Tom Cruise' → 'tomcruise'."""
    return re.sub(r'\s+', '', name.lower().strip())


logger.info('Helper functions defined.')

### 6.2 Apply Feature Extraction

In [ ]:
# Parse columns
logger.info('Parsing genres...')
df['genres_list'] = df['genres'].apply(lambda x: extract_names(x))
df['genres_str'] = df['genres_list'].apply(lambda x: ' '.join([normalize_name(g) for g in x]))

logger.info('Parsing keywords...')
df['keywords_list'] = df['keywords'].apply(lambda x: extract_names(x))
df['keywords_str'] = df['keywords_list'].apply(lambda x: ' '.join([normalize_name(k) for k in x]))

logger.info('Parsing cast (top 3)...')
df['cast_list'] = df['cast'].apply(lambda x: extract_names(x, max_items=TOP_CAST_COUNT))
df['cast_str'] = df['cast_list'].apply(lambda x: ' '.join([normalize_name(c) for c in x]))

logger.info('Parsing director...')
df['director'] = df['crew'].apply(get_director)
df['director_str'] = df['director'].apply(normalize_name)

logger.info('Feature extraction complete.')

### 6.3 Build Combined Metadata Field

In [ ]:
def build_metadata_soup(row: pd.Series) -> str:
    """Combine all metadata into a single text string for TF-IDF."""
    parts = [
        str(row.get('title', '')).lower(),
        str(row.get('overview', '')).lower(),
        str(row.get('genres_str', '')),
        str(row.get('keywords_str', '')),
        str(row.get('cast_str', '')),
        str(row.get('director_str', '')),
        str(int(row.get('year', 0))) if row.get('year', 0) > 0 else ''
    ]
    return ' '.join(parts).strip()


df['metadata_soup'] = df.apply(build_metadata_soup, axis=1)

# Drop rows with empty soup
df = df[df['metadata_soup'].str.strip().astype(bool)].reset_index(drop=True)

logger.info(f'Final dataset shape: {df.shape}')
print('\n--- Sample metadata_soup ---')
print(df['metadata_soup'].iloc[0][:300])

## 7. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(
    stop_words=TFIDF_STOP_WORDS,
    ngram_range=TFIDF_NGRAM_RANGE,
    min_df=TFIDF_MIN_DF,
    max_features=TFIDF_MAX_FEATURES,
    dtype=np.float32  # Save memory
)

logger.info('Fitting TF-IDF vectorizer...')
tfidf_matrix = tfidf.fit_transform(df['metadata_soup'])

print(f'TF-IDF matrix shape : {tfidf_matrix.shape}')
print(f'Vocabulary size     : {len(tfidf.vocabulary_)}')
print(f'Matrix density      : {tfidf_matrix.nnz / (tfidf_matrix.shape[0]*tfidf_matrix.shape[1]):.6f}')
logger.info('TF-IDF vectorization complete.')

## 8. Cosine Similarity Engine

> **Memory Note:** For very large datasets (100k+ movies), building the full N×N similarity 
> matrix is impractical. In that case, compute similarity on-the-fly for a single movie using 
> `linear_kernel(tfidf_matrix[idx], tfidf_matrix)` which returns a 1×N vector.  
> This notebook uses the on-the-fly approach for efficiency.

In [ ]:
# Build a title → index mapping (lowercase for case-insensitive lookup)
df['title_lower'] = df['title'].str.lower().str.strip()
indices = pd.Series(df.index, index=df['title_lower'])

# Handle duplicate titles — keep first occurrence
indices = indices[~indices.index.duplicated(keep='first')]

logger.info(f'Index mapping built: {len(indices)} unique titles.')

## 9. Recommendation Function

In [ ]:
def get_trending_movies(dataframe: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    """Return trending movies by weighted popularity + vote_average (cold start fallback)."""
    temp = dataframe.copy()
    C = temp['vote_average'].mean()
    m = temp['vote_count'].quantile(0.70)
    qualified = temp[temp['vote_count'] >= m].copy()
    qualified['weighted_score'] = (
        (qualified['vote_count'] / (qualified['vote_count'] + m)) * qualified['vote_average']
        + (m / (qualified['vote_count'] + m)) * C
    )
    qualified = qualified.sort_values('weighted_score', ascending=False)
    cols = ['title', 'year', 'genres_str', 'vote_average', 'popularity', 'weighted_score']
    return qualified[cols].head(top_n).reset_index(drop=True)


def explain_match(query_soup: str, candidate_soup: str, top_terms: int = 6) -> str:
    """Generate a simple explainability string showing shared terms."""
    q_tokens = set(query_soup.lower().split())
    c_tokens = set(candidate_soup.lower().split())
    common = q_tokens & c_tokens
    # Filter out very short or generic tokens
    common = sorted([t for t in common if len(t) > 2], key=lambda x: -len(x))
    return ', '.join(common[:top_terms]) if common else 'general thematic similarity'


def recommend_movies(
    title: str,
    top_n: int = DEFAULT_TOP_N,
    show_explanation: bool = True
) -> pd.DataFrame:
    """
    Recommend movies similar to the given title.

    Features:
      - Case-insensitive title matching
      - Fuzzy suggestion if exact title not found
      - Cold-start fallback to trending movies
      - Explainability output showing why each movie matched

    Parameters
    ----------
    title : str
        Movie title to find recommendations for.
    top_n : int
        Number of recommendations to return.
    show_explanation : bool
        Whether to include match explanations.

    Returns
    -------
    pd.DataFrame
        Recommendations with title, year, score, genres, vote_average, popularity.
    """
    title_clean = title.lower().strip()

    # ── Exact match ──
    if title_clean not in indices:
        # ── Fuzzy match ──
        close = get_close_matches(title_clean, indices.index.tolist(), n=5, cutoff=0.6)
        if close:
            print(f'⚠️  "{title}" not found. Did you mean one of these?')
            for c in close:
                print(f'   → {c.title()}')
            # Auto-select best fuzzy match
            title_clean = close[0]
            print(f'\n🔄 Auto-selecting: "{title_clean.title()}"\n')
        else:
            print(f'❌ "{title}" not found and no close matches.')
            print('📈 Showing trending movies instead (Cold Start Fallback):\n')
            return get_trending_movies(df, top_n)

    idx = indices[title_clean]

    # ── Compute similarity on-the-fly (memory efficient) ──
    sim_scores = linear_kernel(tfidf_matrix[idx:idx+1], tfidf_matrix).flatten()
    # Get top indices (exclude self)
    top_indices = sim_scores.argsort()[::-1][1:top_n+1]

    results = []
    query_soup = df.iloc[idx]['metadata_soup']
    for i in top_indices:
        row = df.iloc[i]
        rec = {
            'title': row['title'],
            'year': int(row['year']),
            'score': round(float(sim_scores[i]), 4),
            'genres': row['genres_str'],
            'vote_average': row['vote_average'],
            'popularity': round(float(row['popularity']), 2)
        }
        if show_explanation:
            rec['matched_on'] = explain_match(query_soup, row['metadata_soup'])
        results.append(rec)

    result_df = pd.DataFrame(results)
    print(f'🎬 Top {top_n} recommendations for: "{title_clean.title()}"\n')
    return result_df


logger.info('Recommendation functions ready.')

## 10. Evaluation / Demonstration
Show recommendations for sample movies.

In [ ]:
display(recommend_movies('Avatar', top_n=10))

In [ ]:
display(recommend_movies('The Dark Knight', top_n=10))

In [ ]:
display(recommend_movies('Toy Story', top_n=10))

In [ ]:
display(recommend_movies('Titanic', top_n=10))

## 11. Cold Start Handling Demo

In [ ]:
# Movie that doesn't exist — triggers cold start fallback
display(recommend_movies('NonExistentMovie12345', top_n=10))

## 12. Interactive Lookup (CLI-like)
Change `movie_name` below and re-run to get recommendations.

In [ ]:
# ── Change this variable to get recommendations for any movie ──
movie_name = "Inception"

display(recommend_movies(movie_name, top_n=10, show_explanation=True))

## 13. Save Trained Artifacts
Persist all assets for downstream use in the hybrid pipeline.

In [ ]:
try:
    # Save TF-IDF vectorizer
    joblib.dump(tfidf, os.path.join(ARTIFACT_DIR, 'tfidf_vectorizer.pkl'))
    logger.info('Saved tfidf_vectorizer.pkl')

    # Save TF-IDF matrix
    joblib.dump(tfidf_matrix, os.path.join(ARTIFACT_DIR, 'tfidf_matrix.pkl'))
    logger.info('Saved tfidf_matrix.pkl')

    # Save movie indices mapping
    joblib.dump(indices, os.path.join(ARTIFACT_DIR, 'movie_indices.pkl'))
    logger.info('Saved movie_indices.pkl')

    # Save the processed dataframe as the content model
    model_cols = ['id', 'title', 'title_lower', 'year', 'overview', 'genres_str',
                  'keywords_str', 'cast_str', 'director_str', 'metadata_soup',
                  'vote_average', 'vote_count', 'popularity']
    df_model = df[model_cols].copy()
    joblib.dump(df_model, os.path.join(ARTIFACT_DIR, 'movies_content_model.pkl'))
    logger.info('Saved movies_content_model.pkl')

    print('\n✅ All artifacts saved successfully to:', ARTIFACT_DIR)
    for f in os.listdir(ARTIFACT_DIR):
        fpath = os.path.join(ARTIFACT_DIR, f)
        size_mb = os.path.getsize(fpath) / (1024*1024)
        print(f'   📦 {f:35s} — {size_mb:.2f} MB')

except Exception as e:
    logger.error(f'Failed to save artifacts: {e}')
    raise

## 14. Pipeline Summary

| Step | Detail |
|------|--------|
| Dataset | TMDB movies_metadata + credits + keywords |
| Preprocessing | ID cleaning, null handling, dedup, merge |
| Feature Engineering | genres, keywords, top-3 cast, director, year, overview |
| Vectorization | TF-IDF (unigrams + bigrams, 20k features) |
| Similarity | Cosine similarity via `linear_kernel` (on-the-fly) |
| Cold Start | Weighted popularity fallback |
| Explainability | Shared metadata term analysis |
| Artifacts | 4 `.pkl` files saved for hybrid pipeline integration |

---
**CineIQ Content-Based Filtering Module — Complete ✅**